In [1]:
import pandas as pd
from ast import literal_eval

negation_tokens = {'tidak', 'tak', 'non', 'tanpa'}

df = pd.read_csv('../stemmed_all_v6.csv')
df = df[(df['created_at'] > '2025-10-01') & (df['created_at'] < '2025-11-01')]

df['text'] = df['text'].apply(literal_eval)

from collections import Counter

post_negation = Counter()

for token_list in df['text']:
    for i, token in enumerate(token_list):
        if token in negation_tokens and i + 1 < len(token_list):
            post_negation[token_list[i + 1]] += 1

negation_df = pd.DataFrame(
    post_negation.most_common(),
    columns=['token', 'count']
)
negation_df['antonym'] = ''
negation_df.to_csv('post_negation_tokens_v2.csv', index=False)

print(f"Unique tokens after negation words: {len(post_negation)}")
print(negation_df.head(20))

Unique tokens after negation words: 81
        token  count antonym
0         ada     83        
1      keluar     44        
2       nyala     26        
3        bisa     21        
4        alir     18        
5       punya     10        
6       mandi      9        
7      sampai      7        
8       pakai      4        
9      lancar      4        
10      tidur      3        
11       baik      3        
12      hidup      3        
13      telat      3        
14  informasi      2        
15      warga      2        
16       mati      2        
17       cuci      2        
18      bayar      2        
19      dapat      2        


In [ ]:
def apply_negation_and_synonyms(token_list, negation_tokens, antonym_dict):
    result = []
    skip_next = False

    for i, token in enumerate(token_list):
        if skip_next:
            skip_next = False
            continue

        if token in negation_tokens and i + 1 < len(token_list):
            next_token = token_list[i + 1]
            if next_token in antonym_dict:
                result.append(antonym_dict[next_token])  # replace with antonym
            else:
                # no antonym defined — keep original token, drop negation but merge into one token
                result.append(f'tidak_{next_token}')  # flagged for review
            skip_next = True  # skip the next token since we handled it

        elif token in synonym_dict:
            result.append(synonym_dict[token])  # normalize to canonical form

        else:
            result.append(token)

    return result

antonym_dict = pd.read_csv('./post_negation_tokens.csv').set_index('token')['antonym'].to_dict()

df['tokens_normalized'] = df['text'].apply(
    lambda t: apply_negation_and_synonyms(t, negation_tokens, antonym_dict)
)

df.to_csv('test_antonym.csv', index=False)